In [ ]:
import re
import random
import duckdb
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, BertTokenizer
from sklearn.model_selection import train_test_split
from tqdm import tqdm

1. LOAD DATA

In [ ]:
con = duckdb.connect(database=":memory:")
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

df = con.execute("""
SELECT * FROM read_parquet(
'https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/generation_None_temp08.parquet'
)
""").df()

2. LOAD NACE

In [ ]:
nace = con.execute("""
SELECT * FROM read_csv(
'https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/NACE_Rev2.1_Structure_Explanatory_Notes_EN.tsv'
)
""").df()

nace = nace[nace["CODE"].astype(str).str.len() == 5]
nace = nace[["CODE", "HEADING", "Includes", "IncludesAlso"]]

3. CLEAN TEXT

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"This class include[s]?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"This class also includes?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


nace["Includes"] = nace["Includes"].apply(clean_text)
nace["IncludesAlso"] = nace["IncludesAlso"].apply(clean_text)

nace["nace_text"] = (
    nace["HEADING"].fillna("") + " " +
    nace["Includes"].fillna("") + " " +
    nace["IncludesAlso"].fillna("")
)

4. MERGE DATASET

In [ ]:
df = df.merge(nace[["CODE", "nace_text"]], left_on="code", right_on="CODE")
df = df[["label", "nace_text", "code"]]

5. SPLIT

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["code"],
    random_state=42
)

6. TOKENIZER

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
device = "cuda" if torch.cuda.is_available() else "cpu"

7. MODEL SIAMESE BERT

In [ ]:
class SiameseBERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = BertModel.from_pretrained("bert-base-uncased")
        self.projection = nn.Linear(self.encoder.config.hidden_size, 256)

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        return self.projection(cls)

8. LOSS

In [ ]:
class ContrastiveLoss(nn.Module):
    def forward(self, sim_pos, sim_neg):
        return F.relu(sim_neg - sim_pos + 0.2).mean()

9. NEGATIVES

In [ ]:
all_nace = list(zip(nace["nace_text"], nace["CODE"]))

def get_negative(code):
    while True:
        text, c = random.choice(all_nace)
        if c != code:
            return text

10. TRAINING LOOP 

In [ ]:
model = SiameseBERT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = ContrastiveLoss()

for epoch in range(3):

    total_loss = 0
    loop = tqdm(train_df.itertuples(), total=len(train_df))

    for row in loop:

        optimizer.zero_grad()

        user = tokenizer(row.label, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
        pos = tokenizer(row.nace_text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
        neg = tokenizer(get_negative(row.code), return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

        u = model.encode(user["input_ids"], user["attention_mask"])
        p = model.encode(pos["input_ids"], pos["attention_mask"])
        n = model.encode(neg["input_ids"], neg["attention_mask"])

        sim_pos = F.cosine_similarity(u, p)
        sim_neg = F.cosine_similarity(u, n)

        loss = criterion(sim_pos, sim_neg)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} - loss: {total_loss:.4f}")